# Fine-tune tiếp Detection từ model C trên Colab
Notebook dành cho gói detection_C_colab_input.zip đã chuẩn bị trong workspace. Máy cá nhân không cần GPU.
Chỉ train Detection, không train classifier. Bắt đầu từ exports/det_finetune_best.pt hiện là C.
Đây là một lượt học tiếp theo cấu hình C, không phải D1 và không tái lập chính xác lần B → C cũ.
Dữ liệu gốc được giữ nguyên; xem giới hạn nhãn trong docs/chay-finetune-C-tren-colab.md.

Thứ tự: chọn GPU → upload ZIP lên Drive/fruit_C/input → chạy 1–5 với MODE=smoke.
Smoke thành công: đổi MODE=train ở phần 3, chạy lại phần 3–7.
Không dùng Run all khi chưa đọc các lựa chọn. Checkpoint lưu trực tiếp trên Drive sau mỗi epoch.
Notebook chưa được chạy train GPU trong phiên chuẩn bị.

## 1. GPU và thư viện
Trong cài đặt runtime chọn GPU, T4 nếu được cấp. Nếu không có GPU, đợi hoặc đổi phiên; notebook chủ động dừng.
Chỉ cài Ultralytics đúng phiên bản, không cài lại bộ PyTorch/CUDA của Colab.

In [ ]:
import torch
assert torch.cuda.is_available(), "Chưa có GPU. Chọn GPU runtime; không chạy train trên CPU."
print("GPU:", torch.cuda.get_device_name(0))
%pip install -q ultralytics==8.4.180


## 2. Mount Drive và kiểm tra gói input
Upload detection_C_colab_input.zip vào MyDrive/fruit_C/input trước.
Đọc ảnh từ /content; chỉ model, log và checkpoint ghi vào Drive.
Gói input là phiên bản cố định: resume phải dùng lại đúng ZIP.

In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile
import ultralytics

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/fruit_C")
SOURCE_ZIP = DRIVE_ROOT / "input/detection_C_colab_input.zip"
ROOT = Path("/content/fruit_C")
assert SOURCE_ZIP.is_file(), f"Chưa có file: {SOURCE_ZIP}"
LOCAL_ZIP = Path("/content/detection_C_colab_input.zip")
shutil.copy2(SOURCE_ZIP, LOCAL_ZIP)

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

PACKAGE_SHA = sha256(LOCAL_ZIP)
ROOT.mkdir(parents=True, exist_ok=True)
stamp = ROOT / ".package_sha256"
if stamp.exists():
    assert stamp.read_text() == PACKAGE_SHA, "Đang có dataset của gói khác; dùng runtime mới."
with zipfile.ZipFile(LOCAL_ZIP) as archive:
    for entry in archive.infolist():
        assert (ROOT / entry.filename).resolve().is_relative_to(ROOT.resolve()), "ZIP chứa đường dẫn ngoài ROOT."
    archive.extractall(ROOT)
stamp.write_text(PACKAGE_SHA)
manifest = json.loads((ROOT / "colab_input_manifest.json").read_text(encoding="utf-8"))
assert ultralytics.__version__ == "8.4.180"
assert sha256(ROOT / "exports/det_finetune_best.pt") == manifest["model_sha256"]
assert sha256(ROOT / "work/exp/det_neg.py") == manifest["det_neg_script_sha256"]
# Xác nhận dữ liệu/model đúng gói, không tin các YAML/cache cũ của máy lead.
for item in manifest["files"]:
    path = ROOT / item["path"]
    assert path.is_file() and path.stat().st_size == item["size"], f"Thiếu hoặc sai kích thước: {path}"
    assert sha256(path) == item["sha256"], f"File không khớp manifest: {path}"
print(json.dumps({k: v for k, v in manifest.items() if k != "files"}, indent=2, ensure_ascii=False))
print("Gói ZIP SHA-256:", PACKAGE_SHA)


## 3. Cấu hình lượt chạy
Lượt đầu giữ MODE=smoke để chạy thử 1 epoch. Thành công thì đổi MODE=train và chạy lại phần 3–5.
Hai mode đều khởi tạo từ model C gốc, không dùng model smoke để bắt đầu lượt chính.
Muốn thử một cấu hình mới thì đổi RUN_VERSION; không ghi lẫn vào lượt cũ.
Bị ngắt: chạy phần 1–2, đặt MODE=train, RESUME=True và cùng RUN_VERSION rồi chạy phần 3–5.

In [ ]:
MODE = "smoke"             # "smoke" hoặc "train"
RUN_VERSION = "v1"          # Đổi v2/v3 cho thí nghiệm mới; giữ nguyên khi resume.
BATCH = 16                 # Nếu OOM: 8, rồi 4.
RESUME = False             # True chỉ để tiếp tục lượt bị ngắt.
RESUME_CHECKPOINT = "last.pt"  # Có thể chọn một epoch*.pt được lưu trước đó nếu last.pt hỏng.

assert MODE in {"smoke", "train"}
assert BATCH > 0
assert RUN_VERSION and all(c.isalnum() or c in "-_" for c in RUN_VERSION)
assert Path(RESUME_CHECKPOINT).name == RESUME_CHECKPOINT
EPOCHS = 1 if MODE == "smoke" else 25
RUN_NAME = f"det_C_{'smoke' if MODE == 'smoke' else 'repeat'}_{RUN_VERSION}"
MODEL = ROOT / "exports/det_finetune_best.pt"
PROJECT = DRIVE_ROOT / "runs/exp"
RUN_DIR = PROJECT / RUN_NAME
INPUT_IDENTITY = PROJECT / f"{RUN_NAME}_inputs.json"
NEG_REP = 2
PROJECT.mkdir(parents=True, exist_ok=True)
print({"mode": MODE, "run": RUN_NAME, "epochs": EPOCHS, "batch": BATCH, "checkpoints": str(RUN_DIR)})


## 4. Chuẩn bị đúng tập train của script gốc
Dùng nguyên phần chuẩn bị dữ liệu trong work/exp/det_neg.py; tách phần này ra trước lời gọi train.
Không sửa script gốc. Phần chuẩn bị chỉ chạy trong /content/fruit_C.
Nó loại 50 ảnh quarantine, lấy ảnh own-* từ train_ft nhân 4 và ảnh nền train nhân 2.
Own_cam được chuyển nhãn như script gốc nhưng không được thêm vào danh sách train C.
Các nhãn rỗng/polygon của tập gốc vẫn được giữ nguyên, không tự sửa test.
Ultralytics có thể chuyển polygon thành box khi nạp Detection; cần đọc log về nhãn bị bỏ qua.

In [ ]:
source = (ROOT / "work/exp/det_neg.py").read_text(encoding="utf-8")
marker = "\nYOLO(model).train("
assert source.count(marker) == 1, "Script gốc đã đổi cấu trúc; cần kiểm tra trước khi chạy."
prepare_script = ROOT / "work/exp/prepare_C_colab.py"
prepare_script.write_text(source.split(marker, 1)[0] + "\n", encoding="utf-8")
subprocess.run([sys.executable, str(prepare_script), str(MODEL), RUN_NAME,
                str(EPOCHS), str(BATCH), str(NEG_REP)], cwd=ROOT, check=True)

DATA_YAML = ROOT / f"work/exp/{RUN_NAME}.yaml"
TRAIN_LIST = ROOT / f"work/exp/{RUN_NAME}_train.txt"
train_paths = TRAIN_LIST.read_text().splitlines()
assert len(train_paths) == manifest["training_list"]["total_entries"]
assert all(Path(p).is_file() for p in train_paths)
quarantine = set((ROOT / "work/audit/quarantine_train.txt").read_text().split())
assert not any(Path(p).name in quarantine for p in train_paths)
assert len(list((ROOT / "work/exp/neg/train/images").glob("*.jpg"))) == 287
assert len(list((ROOT / "work/exp/neg/test/images").glob("*.jpg"))) == 71
print("Train entries:", len(train_paths), "| YAML:", DATA_YAML)
print("Dữ liệu vẫn giữ các giới hạn nhãn của bản gốc; không tự thay đổi test.")


## 5. Chạy smoke hoặc train chính — checkpoint lưu trực tiếp Drive
Cấu hình học giữ theo script: SGD, lr0=0.002, warmup=1, seed=42, patience=10, ảnh 640, augmentation HSV như cũ.
Chỉ rõ device=0 để không vô tình train CPU. Workers=2 phù hợp Colab; batch có thể giảm khi thiếu VRAM.
Giữ last.pt để đánh giá theo hướng dẫn lead. Lưu thêm epoch*.pt mỗi 5 epoch để có checkpoint dự phòng.
Resume chỉ dùng khi bị gián đoạn, không dùng sau khi lượt train đã kết thúc.

In [ ]:
from ultralytics import YOLO
import gc

identity = {"package_sha256": PACKAGE_SHA, "model_sha256": manifest["model_sha256"],
            "ultralytics": ultralytics.__version__, "mode": MODE, "run": RUN_NAME,
            "epochs": EPOCHS, "initial_batch": BATCH,
            "recipe": {"imgsz": 640, "neg_rep": 2, "own_rep": 4, "optimizer": "SGD", "lr0": 0.002}}
gc.collect()
torch.cuda.empty_cache()
if RESUME:
    checkpoint = RUN_DIR / "weights" / RESUME_CHECKPOINT
    assert checkpoint.is_file(), f"Chưa có checkpoint: {checkpoint}"
    previous = json.loads(INPUT_IDENTITY.read_text(encoding="utf-8"))
    for key in ("package_sha256", "model_sha256", "ultralytics", "mode", "run", "epochs", "recipe"):
        assert previous[key] == identity[key], f"Không resume khi khác {key}."
    model = YOLO(str(checkpoint))
    model.train(resume=True, data=str(DATA_YAML), device=0, batch=BATCH, workers=2)
else:
    assert not RUN_DIR.exists(), "Run đã tồn tại. Đổi RUN_VERSION hoặc dùng RESUME nếu bị ngắt."
    INPUT_IDENTITY.write_text(json.dumps(identity, indent=2), encoding="utf-8")
    model = YOLO(str(MODEL))
    assert model.task == "detect" and model.names == {0: "banana", 1: "apple", 2: "orange"}
    model.train(data=str(DATA_YAML), epochs=EPOCHS, imgsz=640, batch=BATCH,
                optimizer="SGD", lr0=0.002, warmup_epochs=1, seed=42,
                deterministic=True, patience=10, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
                device=0, workers=2, amp=True, cache=False, save=True, save_period=5,
                project=str(PROJECT), name=RUN_NAME, exist_ok=False, plots=False)
assert Path(model.trainer.save_dir).resolve() == RUN_DIR.resolve(), "Thư mục output khác cấu hình."
assert sha256(MODEL) == manifest["model_sha256"], "Model gốc bị thay đổi."
shutil.copy2(INPUT_IDENTITY, RUN_DIR / "input_identity.json")
print("Đã xong:", RUN_DIR)
if MODE == "smoke":
    print("Smoke đã xong. Đổi MODE=train, RESUME=False, rồi chạy lại phần 3–5.")
else:
    print("Lượt train chính đã hoàn tất. Chạy phần 6 để so sánh C gốc với last.pt, rồi phần 7 để tải model.")
del model
gc.collect()
torch.cuda.empty_cache()


## 6. So baseline C với last.pt sau lượt train chính
Không đánh giá model smoke. Dùng cùng 373 ảnh test và 71 ảnh nền giữ riêng.
mAP mặc định conf=0.001 theo work/eval_det.py; ghi thêm mAP tại conf=0.25 để tránh lẫn hai cách đo.
Ảnh nền dùng conf=0.25, NMS IoU=0.7, như quy trình so thí nghiệm; không dùng ngưỡng web 0.40 ở đây.
Dataset có nhãn lỗi đã biết, nên báo cáo giới hạn này. Không tự chọn model thắng hoặc ghi đè exports.

In [ ]:
import csv

if MODE != "train":
    print("Bỏ qua đánh giá: đây là smoke test, hãy chạy lượt MODE=train trước.")
else:
    held = sorted((ROOT / "work/exp/neg/test/images").glob("*.jpg"))
    assert len(held) == 71
    rows = []
    for tag, weights in (("C_baseline", MODEL), ("C_after_finetune", RUN_DIR / "weights/last.pt")):
        detector = YOLO(str(weights))
        common = dict(data=str(DATA_YAML), split="test", imgsz=640, batch=min(BATCH, 8),
                      device=0, workers=2, iou=0.7, agnostic_nms=False, quantize=16,
                      verbose=False, plots=False, project=str(RUN_DIR / "evaluation"), exist_ok=True)
        low = detector.val(**common, conf=0.001, name=tag + "_map")
        row = {"model": tag, "mAP50": float(low.box.map50), "mAP50_95": float(low.box.map)}
        for c, ap in zip(low.box.ap_class_index, low.box.ap50):
            row["AP50_" + detector.names[int(c)]] = float(ap)
        at25 = detector.val(**common, conf=0.25, name=tag + "_c25")
        row["mAP50_c25"] = float(at25.box.map50)
        row["mAP50_95_c25"] = float(at25.box.map)
        fp = 0
        for i in range(0, len(held), 8):
            for result in detector.predict([str(p) for p in held[i:i+8]], stream=True, imgsz=640,
                        conf=0.25, iou=0.7, agnostic_nms=False, quantize=16, device=0, verbose=False):
                fp += int(len(result.boxes) > 0)
        row.update(background_false_positive_images=fp, background_test_images=len(held))
        rows.append(row)
        print(json.dumps(row, indent=2))
        del detector
        gc.collect()
        torch.cuda.empty_cache()
    fields = list(dict.fromkeys(key for row in rows for key in row))
    with (RUN_DIR / "comparison.csv").open("w", newline="", encoding="utf-8-sig") as stream:
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)
    (RUN_DIR / "comparison.json").write_text(json.dumps(rows, indent=2), encoding="utf-8")
    print("Không mặc định model mới tốt hơn: xem đồng thời mAP, AP từng loại và báo sai nền.")


## 7. Tải model candidate sau lượt train chính
Last.pt vẫn được giữ trên Drive. Xuất tên mới để thử, không ghi đè model hiện tại.
Nếu model mới kém hơn thì giữ C gốc. Classifier không đổi.

In [ ]:
from google.colab import files

if MODE != "train":
    print("Không xuất model smoke để triển khai.")
else:
    OUTPUT = DRIVE_ROOT / "outputs" / RUN_NAME
    OUTPUT.mkdir(parents=True, exist_ok=True)
    candidate = OUTPUT / "det_C_candidate.pt"
    shutil.copy2(RUN_DIR / "weights/last.pt", candidate)
    for name in ("comparison.csv", "comparison.json", "input_identity.json"):
        if (RUN_DIR / name).is_file():
            shutil.copy2(RUN_DIR / name, OUTPUT / name)
    print("Model và báo cáo:", OUTPUT)
    files.download(str(candidate))
